In [1]:
%pip -q install indic-transliteration


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import unicodedata
import pandas as pd
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate


In [3]:
samples = [
    ("Devanagari", "नमस्ते, आप कैसे हैं?"),
    ("Devanagari", "भारत एक विविध देश है।"),
    ("Devanagari", "मुझे हिंदी भाषा पसंद है।"),
    ("Bengali", "নমস্কার, আপনি কেমন আছেন?"),
    ("Bengali", "ভারত একটি বৈচিত্র্যময় দেশ।"),
    ("Bengali", "আমি বাংলা ভাষা পছন্দ করি।")
]

df = pd.DataFrame(samples, columns=["Script", "Original Text"])
df


,Script,Original Text
0,Devanagari,"नमस्ते, आप कैसे हैं?"
1,Devanagari,भारत एक विविध देश है।
2,Devanagari,मुझे हिंदी भाषा पसंद है।
3,Bengali,"নমস্কার, আপনি কেমন আছেন?"
4,Bengali,ভারত একটি বৈচিত্র্যময় দেশ।
5,Bengali,আমি বাংলা ভাষা পছন্দ করি।


In [4]:
df["Unicode Normalized"] = df["Original Text"].apply(
    lambda text: unicodedata.normalize("NFC", text)
)

df


,Script,Original Text,Unicode Normalized
0,Devanagari,"नमस्ते, आप कैसे हैं?","नमस्ते, आप कैसे हैं?"
1,Devanagari,भारत एक विविध देश है।,भारत एक विविध देश है।
2,Devanagari,मुझे हिंदी भाषा पसंद है।,मुझे हिंदी भाषा पसंद है।
3,Bengali,"নমস্কার, আপনি কেমন আছেন?","নমস্কার, আপনি কেমন আছেন?"
4,Bengali,ভারত একটি বৈচিত্র্যময় দেশ।,ভারত একটি বৈচিত্র্যময় দেশ।
5,Bengali,আমি বাংলা ভাষা পছন্দ করি।,আমি বাংলা ভাষা পছন্দ করি।


In [5]:
def detect_script(text):
    devanagari = sum(0x0900 <= ord(ch) <= 0x097F for ch in text)
    bengali = sum(0x0980 <= ord(ch) <= 0x09FF for ch in text)

    if devanagari > bengali:
        return "Devanagari"
    elif bengali > devanagari:
        return "Bengali"
    return "Other"

df["Detected Script"] = df["Unicode Normalized"].apply(detect_script)
df


,Script,Original Text,Unicode Normalized,Detected Script
0,Devanagari,"नमस्ते, आप कैसे हैं?","नमस्ते, आप कैसे हैं?",Devanagari
1,Devanagari,भारत एक विविध देश है।,भारत एक विविध देश है।,Devanagari
2,Devanagari,मुझे हिंदी भाषा पसंद है।,मुझे हिंदी भाषा पसंद है।,Devanagari
3,Bengali,"নমস্কার, আপনি কেমন আছেন?","নমস্কার, আপনি কেমন আছেন?",Bengali
4,Bengali,ভারত একটি বৈচিত্র্যময় দেশ।,ভারত একটি বৈচিত্র্যময় দেশ।,Bengali
5,Bengali,আমি বাংলা ভাষা পছন্দ করি।,আমি বাংলা ভাষা পছন্দ করি।,Bengali


In [6]:
def transliterate_text(text, script):
    if script == "Devanagari":
        return transliterate(text, sanscript.DEVANAGARI, sanscript.ITRANS)
    elif script == "Bengali":
        return transliterate(text, sanscript.BENGALI, sanscript.ITRANS)
    return text

df["Transliterated Text"] = [
    transliterate_text(text, script)
    for text, script in zip(df["Unicode Normalized"], df["Detected Script"])
]

df[["Script", "Original Text", "Unicode Normalized", "Transliterated Text"]]


,Script,Original Text,Unicode Normalized,Transliterated Text
0,Devanagari,"नमस्ते, आप कैसे हैं?","नमस्ते, आप कैसे हैं?","namaste, Apa kaise haiM?"
1,Devanagari,भारत एक विविध देश है।,भारत एक विविध देश है।,bhArata eka vividha desha hai|
2,Devanagari,मुझे हिंदी भाषा पसंद है।,मुझे हिंदी भाषा पसंद है।,mujhe hiMdI bhAShA pasaMda hai|
3,Bengali,"নমস্কার, আপনি কেমন আছেন?","নমস্কার, আপনি কেমন আছেন?","namaskAra, Apani kemana AChena?"
4,Bengali,ভারত একটি বৈচিত্র্যময় দেশ।,ভারত একটি বৈচিত্র্যময় দেশ।,bhArata ekaTi vaichitryamaya় desha|
5,Bengali,আমি বাংলা ভাষা পছন্দ করি।,আমি বাংলা ভাষা পছন্দ করি।,Ami vAMlA bhAShA paChanda kari|
